# **Prediksi Praktikum 09 - IF5100 Pemrograman untuk Data Analitik**
## Machine Learning dengan scikit-learn

Nama : ______________________

NIM  : ______________________

### Aturan / Rules

- Kerjakan di **Google Colab** (runtime CPU). Jalankan sel **berurutan** dari atas ke bawah.
- Library: `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`.
- Gunakan `random_state=42` di **setiap** fungsi yang punya parameter tersebut (`train_test_split`, model, `KFold`, `KMeans`) agar hasil dapat direproduksi.
- **Jangan ubah** nama variabel yang diminta soal dan sel *setup dataset*.
- Setiap grafik wajib punya **judul, label sumbu, dan legend** (jika ada lebih dari satu seri).

| Soal | Topik | Poin |
|---|---|---|
| 1 | Regresi linier dasar & overfitting (data sintetis NumPy) | 20 |
| 2 | Prediksi harga rumah — preprocessing `Pipeline` + `ColumnTransformer` | 30 |
| 3 | Klasifikasi *churn* pelanggan — metrik klasifikasi, KNN, `GridSearchCV` | 30 |
| 4 | Segmentasi pelanggan — `KMeans`, elbow, silhouette | 20 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_auc_score, RocCurveDisplay, silhouette_score)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

---
## SOAL 1 — Regresi Linier Dasar & Overfitting (20 poin)

### Setup data (jangan diubah)

In [ ]:
rng = np.random.default_rng(42)
x = rng.uniform(0, 10, 100)
y = 3 * x + 5 + rng.normal(0, 2, 100)          # hubungan linier + noise

x_poly = np.sort(rng.uniform(-3, 3, 40))
y_poly = 0.5 * x_poly**3 - x_poly**2 - 2 * x_poly + rng.normal(0, 1.5, 40)   # hubungan non-linier

**a.** Ubah `x` menjadi matriks fitur 2D `X` (bentuk `(100, 1)`), lalu bagi menjadi `X_train, X_test, y_train, y_test` dengan proporsi **80:20** dan `random_state=42`. Tampilkan bentuk keempatnya.

*Reshape x into a 2D feature matrix X, split 80:20 with random_state=42, and print the shapes.*

In [ ]:
# TODO

**b.** Latih `LinearRegression` sebagai `model_lr`. Tampilkan koefisien (`coef_`) dan intercept (`intercept_`), lalu tuliskan persamaan garisnya. Bandingkan dengan fungsi asli `y = 3x + 5`.

In [ ]:
# TODO

**c.** Prediksi `X_test`, lalu hitung **MAE, MSE, RMSE, dan R²** (4 desimal).

In [ ]:
# TODO

**d.** Buat scatter plot data train (biru) dan data test (oranye), beserta garis regresi `model_lr` (merah).

In [ ]:
# TODO

**e.** Dengan data `x_poly, y_poly`, bagi 70:30 (`random_state=42`). Untuk derajat polinomial **1, 3, dan 15**, buat pipeline `PolynomialFeatures(degree) → LinearRegression`, hitung **R² train dan R² test**, simpan dalam DataFrame `hasil_poly` (index = derajat). Plot ketiga kurva dalam 1 figure 1×3.

In [ ]:
# TODO

**f.** Berdasarkan `hasil_poly`, derajat mana yang mengalami *underfitting*, *good fit*, dan *overfitting*? Jelaskan dengan membandingkan R² train dan R² test.

**Jawaban:** _tulis di sini_

---
## SOAL 2 — Prediksi Harga Rumah (30 poin)

Sebuah agen properti ingin memperkirakan **harga rumah (juta rupiah)** dari karakteristiknya. Datanya kotor: ada nilai kosong di kolom numerik dan kategorikal.

| Kolom | Keterangan |
|---|---|
| `id_rumah` | ID unik (bukan fitur) |
| `luas_bangunan` | m² |
| `luas_tanah` | m² (ada NaN) |
| `kamar_tidur` | jumlah kamar |
| `umur_bangunan` | tahun (ada NaN) |
| `jarak_pusat_km` | jarak ke pusat kota |
| `kota` | Jakarta / Bandung / Surabaya (ada NaN) |
| `sertifikat` | SHM / HGB |
| `harga_juta` | **target** |

### Setup dataset (jangan diubah)

In [ ]:
def buat_rumah(seed=9, n=600):
    rng = np.random.default_rng(seed)
    lb = rng.uniform(36, 300, n).round()
    lt = (lb * rng.uniform(1.0, 2.0, n)).round()
    kamar = np.clip(np.round(lb / 45 + rng.normal(0, 0.7, n)), 1, 7).astype(int)
    umur = rng.integers(0, 41, n).astype(float)
    jarak = rng.gamma(2.0, 4.0, n).round(1)
    kota = rng.choice(["Jakarta", "Bandung", "Surabaya"], n, p=[0.4, 0.35, 0.25])
    sert = rng.choice(["SHM", "HGB"], n, p=[0.7, 0.3])
    premium = pd.Series({"Jakarta": 900, "Bandung": 300, "Surabaya": 200})[kota].to_numpy()
    harga = (400 + 6 * lb + 2.5 * lt + 60 * kamar - 8 * umur - 25 * jarak + premium
             + np.where(sert == "SHM", 150, 0) + rng.normal(0, 180, n)).round()
    df = pd.DataFrame({"id_rumah": [f"R{i:04d}" for i in range(1, n + 1)],
                       "luas_bangunan": lb, "luas_tanah": lt, "kamar_tidur": kamar,
                       "umur_bangunan": umur, "jarak_pusat_km": jarak, "kota": kota,
                       "sertifikat": sert, "harga_juta": harga})
    for col, frac in [("luas_tanah", 0.05), ("umur_bangunan", 0.03), ("kota", 0.04)]:
        df.loc[rng.choice(n, int(frac * n), replace=False), col] = np.nan
    return df


df_rumah = buat_rumah()
df_rumah.head()

**a.** Tampilkan ukuran, tipe data, jumlah *missing value* per kolom, dan statistik deskriptif `harga_juta`.

In [ ]:
# TODO

**b.** Pisahkan fitur `X_rumah` (semua kolom kecuali `id_rumah` dan `harga_juta`) dan target `y_rumah`. Bagi 80:20 (`random_state=42`) menjadi `Xr_train, Xr_test, yr_train, yr_test`.

Buat juga list `fitur_numerik` dan `fitur_kategorikal`.

In [ ]:
# TODO

**c.** Buat `preprocessor` dengan `ColumnTransformer`:
- numerik → `SimpleImputer(strategy="median")` lalu `StandardScaler()`
- kategorikal → `SimpleImputer(strategy="most_frequent")` lalu `OneHotEncoder(handle_unknown="ignore")`

Gabungkan dengan `LinearRegression` dalam `Pipeline` bernama `pipe_lr`, latih di data train, lalu tampilkan **MAE, RMSE, R²** pada data test.

In [ ]:
# TODO

**d.** Ganti model dengan `RandomForestRegressor(n_estimators=200, random_state=42)` (preprocessor sama) sebagai `pipe_rf`. Buat DataFrame `perbandingan` berisi MAE, RMSE, R² kedua model.

In [ ]:
# TODO

**e.** Lakukan **5-fold cross-validation** (`KFold(n_splits=5, shuffle=True, random_state=42)`, `scoring="r2"`) untuk kedua pipeline pada **seluruh** `X_rumah, y_rumah`. Tampilkan skor tiap fold, rata-rata, dan standar deviasinya.

In [ ]:
# TODO

**f.** Buat 2 panel: (1) scatter **harga aktual vs prediksi** `pipe_lr` dengan garis y = x; (2) histogram **residual** (aktual − prediksi).

In [ ]:
# TODO

**g.** Ambil nama fitur hasil preprocessing (`get_feature_names_out()`) dan koefisien `pipe_lr`. Tampilkan sebagai **horizontal bar chart** terurut. Fitur apa yang paling berpengaruh?

In [ ]:
# TODO

**h.** (i) Mengapa `fit` imputer/scaler hanya boleh dilakukan pada data train? (ii) Model mana yang lebih baik pada soal ini dan mengapa? (iii) Interpretasikan koefisien terbesar pada butir g.

**Jawaban:** _tulis di sini_

---
## SOAL 3 — Klasifikasi Churn Pelanggan (30 poin)

Perusahaan internet **NusaNet** ingin memprediksi pelanggan yang akan berhenti berlangganan (*churn*).

| Kolom | Keterangan |
|---|---|
| `lama_langganan` | bulan |
| `biaya_bulanan` | ribu rupiah |
| `jumlah_keluhan` | 3 bulan terakhir |
| `kecepatan_mbps` | paket kecepatan |
| `kontrak` | Bulanan / Tahunan / Dua Tahun |
| `metode_bayar` | Transfer / E-Wallet / Kartu Kredit |
| `churn` | **target** (1 = berhenti, 0 = bertahan) |

### Setup dataset (jangan diubah)

In [ ]:
def buat_churn(seed=10, n=1500):
    rng = np.random.default_rng(seed)
    kontrak = rng.choice(["Bulanan", "Tahunan", "Dua Tahun"], n, p=[0.5, 0.3, 0.2])
    lama = np.where(kontrak == "Bulanan", rng.integers(1, 30, n), rng.integers(6, 73, n))
    kecepatan = rng.choice([20, 50, 100, 300], n, p=[0.35, 0.35, 0.2, 0.1])
    biaya = (150 + kecepatan * 0.9 + rng.normal(0, 40, n)).round()
    keluhan = rng.poisson(0.8, n)
    metode = rng.choice(["Transfer", "E-Wallet", "Kartu Kredit"], n, p=[0.4, 0.4, 0.2])
    efek_kontrak = pd.Series({"Bulanan": 0.9, "Tahunan": -0.8, "Dua Tahun": -1.8})[kontrak].to_numpy()
    logit = (-1.6 - 0.04 * lama + 0.006 * (biaya - 200) + 0.75 * keluhan + efek_kontrak
             + np.where(metode == "E-Wallet", 0.3, 0))
    churn = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return pd.DataFrame({"lama_langganan": lama, "biaya_bulanan": biaya, "jumlah_keluhan": keluhan,
                         "kecepatan_mbps": kecepatan, "kontrak": kontrak, "metode_bayar": metode,
                         "churn": churn})


df_churn = buat_churn()
df_churn.head()

**a.** Tampilkan distribusi kelas `churn` (jumlah dan persentase) dan visualisasikan dengan bar chart. Tampilkan juga **churn rate per jenis kontrak**.

In [ ]:
# TODO

**b.** Pisahkan `X_churn`, `y_churn`. Bagi 75:25 dengan **`stratify=y_churn`** dan `random_state=42` menjadi `Xc_train, Xc_test, yc_train, yc_test`. Buktikan proporsi kelas train dan test hampir sama.

In [ ]:
# TODO

**c.** Buat `prep_churn` (`StandardScaler` untuk numerik, `OneHotEncoder(handle_unknown="ignore")` untuk kategorikal) dan pipeline `pipe_logreg` dengan `LogisticRegression(max_iter=1000)`. Tampilkan **accuracy, precision, recall, F1** (kelas positif = 1), `classification_report`, dan **confusion matrix** (`ConfusionMatrixDisplay`).

Bandingkan juga dengan **baseline** yang selalu menebak kelas mayoritas (0).

In [ ]:
# TODO

**d.** Bank ingin menangkap lebih banyak pelanggan churn. Gunakan `predict_proba` dan ubah **threshold** dari 0.5 menjadi **0.3**. Bandingkan precision, recall, dan F1 kedua threshold. Tampilkan juga **ROC curve dan AUC**.

In [ ]:
# TODO

**e.** Latih `KNeighborsClassifier` untuk **k = 1, 3, 5, …, 29**, **dengan** dan **tanpa** scaling (tanpa scaling: fitur numerik dipakai apa adanya, kategorikal tetap di-one-hot). Plot **F1 test** terhadap k untuk kedua versi dalam satu grafik.

In [ ]:
# TODO

**f.** Gunakan `GridSearchCV` (5-fold `StratifiedKFold`, `scoring="f1"`) untuk mencari `max_depth` ∈ {2, 3, 4, 5, 6, 8, 10, 15} dan `min_samples_leaf` ∈ {1, 5, 10, 20} pada `DecisionTreeClassifier(random_state=42)` (dengan `prep_churn`). Tampilkan parameter terbaik, skor CV terbaik, dan F1 di data test.

In [ ]:
# TODO

**g.** (i) Mengapa accuracy saja menyesatkan pada data ini? (ii) Untuk kasus churn, mana yang lebih penting: precision atau recall? Kaitkan dengan hasil butir d. (iii) Mengapa KNN sangat terpengaruh scaling?

**Jawaban:** _tulis di sini_

---
## SOAL 4 — Segmentasi Pelanggan dengan K-Means (20 poin)

Sebuah mal ingin mengelompokkan pengunjung berdasarkan perilaku belanja (tanpa label → *unsupervised learning*).

### Setup dataset (jangan diubah)

In [ ]:
def buat_mall(seed=11):
    rng = np.random.default_rng(seed)
    pusat = {"pendapatan_juta": [8, 25, 25, 60, 60], "skor_belanja": [20, 50, 85, 15, 80],
             "umur": [45, 38, 24, 42, 30], "kunjungan_bulanan": [2, 4, 9, 2, 7]}
    sebaran = {"pendapatan_juta": 4, "skor_belanja": 7, "umur": 8, "kunjungan_bulanan": 1.2}
    ukuran = [60, 80, 50, 50, 60]
    blok = []
    for i, n in enumerate(ukuran):
        blok.append(pd.DataFrame({c: rng.normal(pusat[c][i], sebaran[c], n) for c in pusat}))
    df = pd.concat(blok, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    df["pendapatan_juta"] = df["pendapatan_juta"].clip(3).round(1)
    df["skor_belanja"] = df["skor_belanja"].clip(1, 100).round()
    df["umur"] = df["umur"].clip(17, 70).round()
    df["kunjungan_bulanan"] = df["kunjungan_bulanan"].clip(0).round()
    df.insert(0, "id_pelanggan", [f"C{i:03d}" for i in range(1, len(df) + 1)])
    return df


df_mall = buat_mall()
df_mall.head()

**a.** Ambil fitur `pendapatan_juta, skor_belanja, umur, kunjungan_bulanan`, standardisasi dengan `StandardScaler` menjadi `X_mall_scaled`. Tampilkan mean dan std tiap kolom setelah scaling.

In [ ]:
# TODO

**b.** Untuk k = 2 sampai 10, latih `KMeans(n_clusters=k, n_init=10, random_state=42)`, simpan **inertia** dan **silhouette score**. Plot keduanya dalam figure 1×2 (*elbow method* & silhouette). Tentukan `k_terbaik`.

In [ ]:
# TODO

**c.** Latih KMeans final dengan `k_terbaik`, simpan label ke kolom `cluster` di `df_mall`. Buat **profil cluster**: rata-rata tiap fitur dan jumlah anggota per cluster.

In [ ]:
# TODO

**d.** Visualisasikan hasil clustering: (1) scatter `pendapatan_juta` vs `skor_belanja` diwarnai cluster, beserta **centroid** (kembalikan ke skala asli dengan `scaler_mall.inverse_transform`); (2) scatter 2 komponen utama **PCA** diwarnai cluster.

In [ ]:
# TODO

**e.** Beri nama tiap segmen berdasarkan profilnya dan satu rekomendasi strategi pemasaran. Mengapa data harus distandardisasi sebelum K-Means?

**Jawaban:** _tulis di sini_

---
## Soal Teori

### 1. Apa beda *supervised* dan *unsupervised learning*? Beri contoh masing-masing dari praktikum ini.

**Jawaban:** _tulis di sini_

### 2. Jelaskan perbedaan `fit`, `transform`, `fit_transform`, dan `predict`.

**Jawaban:** _tulis di sini_

### 3. Apa itu *data leakage*? Sebutkan 2 contohnya.

**Jawaban:** _tulis di sini_

### 4. Mengapa perlu `stratify` saat `train_test_split` pada klasifikasi?

**Jawaban:** _tulis di sini_

### 5. Apa fungsi *cross-validation* dibanding satu kali train-test split?

**Jawaban:** _tulis di sini_

### 6. Jelaskan *bias–variance trade-off* dan kaitannya dengan underfitting/overfitting.

**Jawaban:** _tulis di sini_

### 7. Kapan memakai MAE vs RMSE? Apa arti R² = 0 dan R² negatif?

**Jawaban:** _tulis di sini_

### 8. Mengapa `OneHotEncoder` lebih tepat daripada `LabelEncoder` untuk fitur nominal seperti `kota`?

**Jawaban:** _tulis di sini_